In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week5-assignment-1"). \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
## path : /public/trendytech/groceries.csv

In [3]:
groceries_df = spark.read.csv('/public/trendytech/groceries.csv',header = "true",inferSchema = "true")

In [4]:
groceries_df.createOrReplaceTempView("groceries")

In [16]:
groceries_df.count()

21

In [5]:
spark.sql("use itv024128")

""


In [80]:
spark.sql("show tables")

database,tableName,isTemporary
itv024128,groceries,false
itv024128,groceries_ext,false
itv024128,groceries_ext_json,false
itv024128,groceries_json,false
itv024128,orders_ext,false
,groceries,true
,groceries_dfjson,true
,groceries_json,true


### Managed table

In [11]:
spark.sql("create table if not exists itv024128.groceries (order_id string, location string, item string, order_date string , quantity int)")

""


In [14]:
spark.sql("insert into itv024128.groceries (select * from groceries)")

""


In [15]:
spark.sql("select count(*) from itv024128.groceries")

count(1)
21


In [22]:
spark.sql("describe extended itv024128.groceries").show(truncate=False)

+----------------------------+-----------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                    |comment|
+----------------------------+-----------------------------------------------------------------------------+-------+
|order_id                    |string                                                                       |null   |
|location                    |string                                                                       |null   |
|item                        |string                                                                       |null   |
|order_date                  |string                                                                       |null   |
|quantity                    |int                                                                          |null   |
|                            |                                  

In [24]:
spark.sql("drop table itv024128.groceries_ext ")

""


### External table

In [26]:
spark.sql("""
create table if not exists itv024128.groceries_ext 
(order_id string, location string, item string, order_date string , quantity int) 
using csv options(header='true') 
location '/public/trendytech/groceries.csv' """)

""


In [27]:
spark.sql("describe extended itv024128.groceries_ext").show(truncate=False)

+----------------------------+-------------------------------------------------------------+-------+
|col_name                    |data_type                                                    |comment|
+----------------------------+-------------------------------------------------------------+-------+
|order_id                    |string                                                       |null   |
|location                    |string                                                       |null   |
|item                        |string                                                       |null   |
|order_date                  |string                                                       |null   |
|quantity                    |int                                                          |null   |
|                            |                                                             |       |
|# Detailed Table Information|                                                             

In [28]:
spark.sql("select count(*) from itv024128.groceries_ext")

count(1)
21


## from JSON FILE

In [30]:
## (path : /public/trendytech/orders_wh.json/part-00000-68544d18-9a34-443f-bf0e-1dd8103ff94e-c000.json)

In [43]:
groceries_dfjson = spark.read.json('/public/trendytech/orders_wh.json/part-00000-68544d18-9a34-443f-bf0e-1dd8103ff94e-c000.json')

In [44]:
groceries_dfjson.count()

68883

In [45]:
groceries_dfjson.show(5)

+-----------+--------------------+--------+---------------+
|customer_id|          order_date|order_id|   order_status|
+-----------+--------------------+--------+---------------+
|      11599|2013-07-25 00:00:...|       1|         CLOSED|
|        256|2013-07-25 00:00:...|       2|PENDING_PAYMENT|
|      12111|2013-07-25 00:00:...|       3|       COMPLETE|
|       8827|2013-07-25 00:00:...|       4|         CLOSED|
|      11318|2013-07-25 00:00:...|       5|       COMPLETE|
+-----------+--------------------+--------+---------------+
only showing top 5 rows



In [66]:
groceries_dfjson.count()

68883

In [46]:
groceries_dfjson.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_id: long (nullable = true)
 |-- order_status: string (nullable = true)



In [67]:
groceries_dfjson.createOrReplaceTempView("groceries_dfjson")

In [68]:
spark.sql("select * from groceries_dfjson limit (5)")

customer_id,order_date,order_id,order_status
7597,2014-07-07 00:00:...,54917,COMPLETE
11097,2014-07-07 00:00:...,54918,COMPLETE
8237,2014-07-07 00:00:...,54919,CLOSED
4893,2014-07-07 00:00:...,54920,COMPLETE
10810,2014-07-07 00:00:...,54921,ON_HOLD


In [69]:
spark.sql("select count(*) from groceries_dfjson ")

count(1)
68883


""


In [70]:
spark.sql("drop table  itv024128.groceries_json ")

""


In [73]:
spark.sql("create table if not exists itv024128.groceries_json (customer_id long, order_date string, order_id long, order_status string )")

""


In [74]:
spark.sql("insert into itv024128.groceries_json (select * from groceries_dfjson)")

""


In [75]:
spark.sql("select count(*) from itv024128.groceries_json ")

count(1)
68883


In [76]:
spark.sql("drop table  itv024128.groceries_ext_json ")

""


In [77]:
spark.sql("""
create table if not exists itv024128.groceries_ext_json 
(customer_id long, order_date string, order_id long, order_status string )
using json 
location '/public/trendytech/orders_wh.json/part-00000-68544d18-9a34-443f-bf0e-1dd8103ff94e-c000.json' """)

""


In [78]:
spark.sql("select * from itv024128.groceries_ext_json limit (5)")

customer_id,order_date,order_id,order_status
7597,2014-07-07 00:00:...,54917,COMPLETE
11097,2014-07-07 00:00:...,54918,COMPLETE
8237,2014-07-07 00:00:...,54919,CLOSED
4893,2014-07-07 00:00:...,54920,COMPLETE
10810,2014-07-07 00:00:...,54921,ON_HOLD


In [79]:
spark.sql("select count(*) from itv024128.groceries_ext_json ")

count(1)
68883
